# 🌿 FieldPlant Disease Detection — YOLO11m Training

**Model**: YOLO11m (upgraded from YOLOv8m — better architecture, same speed)  
**Dataset**: FieldPlant (30 classes, 15,468 images)  

### ⚡ Checkpoint & Resume Strategy
Kaggle sessions expire after ~12hrs. This notebook handles it:
1. **First run**: Trains from scratch, saves checkpoint every epoch
2. **Session expires**: Kaggle auto-saves `/kaggle/working/` as Output
3. **Next session**: Save previous Output as a Dataset → attach it → resume training

### 📋 Resume Steps (if session expired)
1. Go to your expired notebook → **Output** tab → **New Dataset** → name it `yolo-checkpoint`
2. Open a new notebook → attach `yolo-checkpoint` + `fieldplant-efv6g` datasets
3. Set `CHECKPOINT_DATASET = 'yolo-checkpoint'` in Cell 2
4. Run all cells — it auto-resumes from last epoch

---
## 📦 Cell 1: Setup

In [ ]:
import os, sys, shutil, glob

!pip install -q ultralytics pydantic-settings python-dotenv albumentations

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
num_gpus = torch.cuda.device_count()
for i in range(num_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

print('\n📂 /kaggle/input:')
for d in sorted(os.listdir('/kaggle/input')):
    print(f'  📁 {d}')

---
## 🔧 Cell 2: Configure Paths & Checkpoint
👉 **Update names to match your Kaggle datasets. Set CHECKPOINT_DATASET if resuming.**

In [ ]:
# ============================================================
#  👇 UPDATE THESE
# ============================================================
DATASET_NAME        = 'fieldplant-efv6g'      # Your image dataset
CHECKPOINT_DATASET  = ''                       # SET THIS TO RESUME! e.g. 'yolo-checkpoint'
# ============================================================

DATASET_PATH = os.path.join('/kaggle/input', DATASET_NAME)
WORKING_DIR  = '/kaggle/working'

assert os.path.exists(DATASET_PATH), f'❌ Dataset not found: {DATASET_PATH}'
print(f'✅ Dataset: {DATASET_PATH}')

# ── Check for checkpoint to resume from ──
RESUME_PATH = None
if CHECKPOINT_DATASET:
    ckpt_dir = os.path.join('/kaggle/input', CHECKPOINT_DATASET)
    if os.path.exists(ckpt_dir):
        # Find last.pt anywhere inside the checkpoint dataset
        for root, dirs, files in os.walk(ckpt_dir):
            if 'last.pt' in files:
                # Copy to working dir (input is read-only)
                src = os.path.join(root, 'last.pt')
                # Recreate the expected directory structure
                rel = os.path.relpath(root, ckpt_dir)
                dst_dir = os.path.join(WORKING_DIR, rel)
                os.makedirs(dst_dir, exist_ok=True)
                dst = os.path.join(dst_dir, 'last.pt')
                shutil.copy2(src, dst)
                RESUME_PATH = dst
                print(f'✅ Found checkpoint: {src}')
                print(f'   Copied to: {dst}')
                break
        
        # Also copy best.pt if it exists
        for root, dirs, files in os.walk(ckpt_dir):
            if 'best.pt' in files:
                src = os.path.join(root, 'best.pt')
                rel = os.path.relpath(root, ckpt_dir)
                dst_dir = os.path.join(WORKING_DIR, rel)
                os.makedirs(dst_dir, exist_ok=True)
                shutil.copy2(src, os.path.join(dst_dir, 'best.pt'))
        
        if RESUME_PATH is None:
            print(f'⚠️ Checkpoint dataset exists but no last.pt found inside.')
    else:
        print(f'⚠️ Checkpoint dataset not found: {ckpt_dir}')

if RESUME_PATH:
    print(f'\n🔄 RESUME MODE: Will continue from {RESUME_PATH}')
else:
    print(f'\n🆕 FRESH START: Training from scratch')

---
## 📋 Cell 3: Build data.yaml

In [ ]:
import yaml

# Find data.yaml
original_yaml = None
for root, dirs, files in os.walk(DATASET_PATH):
    if 'data.yaml' in files:
        original_yaml = os.path.join(root, 'data.yaml')
        break

with open(original_yaml, 'r') as f:
    data = yaml.safe_load(f)

# Build with absolute paths
new_data = {'nc': data.get('nc', 30), 'names': data.get('names', [])}

for folder, key in [('train','train'), ('valid','val'), ('val','val'), ('test','test')]:
    p = os.path.join(DATASET_PATH, folder, 'images')
    if os.path.exists(p):
        new_data[key] = os.path.abspath(p)
        print(f'  ✅ {key}: {p} ({len(os.listdir(p))} images)')

if 'val' not in new_data and 'train' in new_data:
    new_data['val'] = new_data['train']
    print('  ⚠️ No val split, using train as val')

KAGGLE_YAML = os.path.join(WORKING_DIR, 'data.yaml')
with open(KAGGLE_YAML, 'w') as f:
    yaml.dump(new_data, f, default_flow_style=False)

print(f'\n✅ Saved: {KAGGLE_YAML}')

---
## 🚀 Cell 4: Train YOLO11m (with Resume Support)

**Why YOLO11m over YOLOv8m?**
- Same parameter count (~25M)
- Better C3k2 backbone + SPPF neck
- ~2-3% higher mAP on benchmarks
- Native support in ultralytics ≥ 8.3

In [ ]:
from ultralytics import YOLO

# ── GPU Config ──
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    DEVICE = [0, 1]
    BATCH  = 32
elif num_gpus == 1:
    DEVICE = 0
    BATCH  = 16
else:
    DEVICE = 'cpu'
    BATCH  = 8

print(f'Device: {DEVICE} | Batch: {BATCH}')

# ── Load Model ──
if RESUME_PATH:
    print(f'\n🔄 Resuming from checkpoint: {RESUME_PATH}')
    model = YOLO(RESUME_PATH)
else:
    print(f'\n🆕 Starting fresh with yolo11m.pt')
    model = YOLO('yolo11m.pt')

# ── Train ──
results = model.train(
    data=KAGGLE_YAML,
    epochs=100,               # Total epochs (resume picks up where it left off)
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    workers=4,
    project=os.path.join(WORKING_DIR, 'runs', 'detect'),
    name='fieldplant_yolo11m',
    exist_ok=True,
    resume=bool(RESUME_PATH),  # Key flag: tells YOLO to resume optimizer + scheduler state
    patience=15,               # Early stopping
    save=True,                 # Saves last.pt every epoch (for resume)
    save_period=5,             # Save checkpoint every 5 epochs as backup
    plots=True,
    verbose=True,
    amp=True,
    cache='ram',
    cos_lr=True,
)

print('\n🎉 Training complete!')

---
## 📊 Cell 5: View Results

In [ ]:
import IPython.display as ipd

runs_dir = os.path.join(WORKING_DIR, 'runs', 'detect')
if os.path.exists(runs_dir):
    folders = sorted(
        [d for d in os.listdir(runs_dir) if os.path.isdir(os.path.join(runs_dir, d))],
        key=lambda x: os.path.getmtime(os.path.join(runs_dir, x)), reverse=True
    )
    if folders:
        results_dir = os.path.join(runs_dir, folders[0])
        print(f'📂 Results: {results_dir}')
        for fname, label in [('results.png','📈 Curves'), ('confusion_matrix.png','🔢 CM'),
                              ('val_batch0_pred.jpg','🖼️ Preds')]:
            fp = os.path.join(results_dir, fname)
            if os.path.exists(fp):
                print(f'\n{label}:')
                ipd.display(ipd.Image(filename=fp, width=800))

---
## 🧪 Cell 6: Test Inference

In [ ]:
import matplotlib.pyplot as plt
import cv2, random

best_pt = os.path.join(results_dir, 'weights', 'best.pt')
trained = YOLO(best_pt) if os.path.exists(best_pt) else model

sample_dir = None
for s in ['valid','val','test','train']:
    c = os.path.join(DATASET_PATH, s, 'images')
    if os.path.exists(c): sample_dir = c; break

if sample_dir:
    imgs = [f for f in os.listdir(sample_dir) if f.lower().endswith(('.jpg','.png'))]
    chosen = random.sample(imgs, min(4, len(imgs)))
    fig, axes = plt.subplots(1, len(chosen), figsize=(6*len(chosen), 6))
    if len(chosen)==1: axes=[axes]
    for ax, name in zip(axes, chosen):
        r = trained.predict(os.path.join(sample_dir, name), save=False, conf=0.25)
        ax.imshow(cv2.cvtColor(r[0].plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(name[:25]); ax.axis('off')
    plt.tight_layout(); plt.show()

---
## 💾 Cell 7: Save Weights for Download

### ⚠️ IMPORTANT: If session is about to expire
1. Don't worry — `last.pt` is already saved in `/kaggle/working/runs/`
2. After expiry: go to **Output** tab → click **"New Dataset"** → name it `yolo-checkpoint`
3. Start new notebook → attach `yolo-checkpoint` → set `CHECKPOINT_DATASET = 'yolo-checkpoint'`
4. Run again — it resumes automatically!

In [ ]:
# Copy best weights to root for easy download
for fname in ['best.pt', 'last.pt']:
    src = os.path.join(results_dir, 'weights', fname)
    dst = os.path.join(WORKING_DIR, f'yolo11m_fieldplant_{fname}')
    if os.path.exists(src):
        shutil.copy2(src, dst)
        mb = os.path.getsize(dst) / (1024*1024)
        print(f'✅ {fname} → {dst} ({mb:.1f} MB)')

print('\n📥 Download from Output tab (right sidebar).')
print('\n🔄 To resume later:')
print('   1. Output tab → New Dataset → name: yolo-checkpoint')
print('   2. New notebook → attach yolo-checkpoint')
print('   3. Set CHECKPOINT_DATASET = "yolo-checkpoint" in Cell 2')